# De Levenscyclus van een Zaak

**Thema:** Dossiers en parlementaire zaken.

**Kernvragen:** Wat zijn de verschillende soorten zaken (`zaak.soort`), hoe lang duurt het voordat een zaak wordt afgedaan (`afgedaan = true`), en welke zaken zijn gemarkeerd als groot project?

Dit notebook verkent o.a. de silver-tabellen `zaak`, `zaak_activiteit`, `zaak_gerelateerd_vanuit` en `zaak_vervangen_vanuit`.

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd
import matplotlib.pyplot as plt

def find_database(start: Path | None = None) -> Path:
    """Find the repository DuckDB database by walking up from the current folder."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        candidate = folder / "tweedekamer.duckdb"
        if candidate.is_file():
            return candidate
    raise FileNotFoundError("Could not find tweedekamer.duckdb from the current directory")


database_path = find_database()
connection = duckdb.connect(str(database_path), read_only=True)


def query_df(sql: str, parameters=None) -> pd.DataFrame:
    """Execute a SQL query and return the result as a pandas DataFrame."""
    return connection.execute(sql, parameters or []).fetchdf()


database_path

## Onderwerp 1

laat de feitentabel zien <br/>
hoeveel zaken worden er afgedaan en niet afgedaan? <br/>
Wat zijn de top 3 zaken die het meest worden afgewezen? <br/>

In [ ]:
df = query_df("""
SELECT *
FROM silver.zaak""")
df

In [ ]:
df = query_df("""
select count(id)
from silver.zaak
where afgedaan = True
""")
df

In [ ]:
df = query_df("""
select count(id)
from silver.zaak
where afgedaan = False
""")
df

In [ ]:
df = query_df(""" 
select soort, titel
from silver.zaak
where afgedaan = False""")
df

In [ ]:
df = query_df("""
select count(*) as "aantal zaken zonder titel"
from silver.zaak
where titel IS NULL""")
df

In [ ]:
df = query_df("""
select count(*) as "hoeveelheid zaken"
from silver.zaak""")
df

In [ ]:
df = query_df("""
select count(soort), soort
from silver.zaak
where afgedaan = False
group by soort
order by count(soort) desc""")
df

In [ ]:
df = query_df("""
select count(soort), soort
from silver.zaak
group by soort
order by count(soort) desc""")
df

In [ ]:
113329/303097

### Conclusie Onderwerp 1
De enige opmerking die er gemaakt kan worden met de data van hierboven is dat er niet bij alle zaken een titel zit, dit geldt voor ongeveer 37% van de zaken. <br/> Voor de rest is er ook antwoord gegeven op de vraag: "Wat zijn de top 3 zaken die het meest worden afgewezen?", namelijk: <br/> <br/>Brief regering <br/> Brief commissie </br> Motie <br/><br/> dit komt natuurlijk doordat er ook meer van die zaken zijn, want er zijn een stuk minder zaken met de soort "Netwerkverkenning", wat wel logisch is.

## Onderwerp 2

Bij dit onderwerp wil ik het resultaat van het vorige onderwerp verbeteren, hiermee wil ik de vraag "Wat zijn de top 3 zaken die het meest worden afgewezen?" vooral verbeteren door te kijken naar welke het meest wordt afgewezen als ik rekening hou met hoeveel zaken er in totaal zijn.
<br/>Bijvoorbeeld als er in totaal 201 brieven waren van de eerste kamer en in totaal maar 1 is afgewezen, dan worden er 0.5% van de zaken niet afgedaan bij deze soort.

er missen een aantal soorten in de tabel van alle zaken die zijn afgewezen, dit komt omdat die zaken altijd zijn afgedaan, dit zijn:

In [ ]:
df = query_df("""
select distinct soort
from silver.zaak
where soort not in 
    (select soort
    from silver.zaak
    where afgedaan = False)
""")
df

met de query hierboven kunnen we dan vervolgens een tabel maken met de hoeveelheid zaken in totaal terwijl het de zaken uitsluit die altijd zijn afgedaan.

In [ ]:
dftotaal = query_df("""
select count(soort) as aantal, soort
from silver.zaak
where soort not in 
    (select distinct soort
    from silver.zaak
    where soort not in 
        (select soort
        from silver.zaak
        where afgedaan = False))
group by soort
order by soort""")
dftotaal

In [ ]:
dfafgewezen = query_df("""
select count(soort) as aantal, soort
from silver.zaak
where afgedaan = False
group by soort
order by soort""")
dfafgewezen

In [ ]:
df = query_df("""
SELECT a.soort, a.aantal AS aantal_afgewezen, t.aantal AS aantal_totaal, ROUND(100.0 * a.aantal / t.aantal, 2) AS percentage_afgewezen
FROM dfafgewezen a
JOIN dftotaal t
ON a.soort = t.soort
ORDER BY percentage_afgewezen desc
""")
df

nu er percentages zijn berekend in plaats van het totaal te pakken, is er een ander antwoord op de vraag "Wat zijn de top 3 zaken die het meest worden afgewezen?" gevonden <br/>
de top 3 zaken die het meest worden afgewezen zijn: <br/>
Initiatiefwetgeving <br/>
Wetenschapstoets <br/>
Overig (openbaar) <br/> <br/>
hier komt wel weer bij dat het totaal van deze soorten zaken wel laag kan liggen, dus het is waarschijnlijk beter om een ondergrens te nemen van tenminste 100 zaken, hierbij bedoel ik dat een soort zaak minstens 100 ervan moet hebben om zich te kwalificeren, dus dan worden de top 3 meest afgewezen zaken:<br/>
Initiatiefwetgeving <br/>
Initiatiefnota <br/>
Wetenschappelijke factsheet <br/>

In [ ]:
df = query_df("""
select count(soort), soort
from silver.zaak
where groot_project = True
group by soort
""")
df